In [1]:
import pandas as pd
import time
import requests
from io import StringIO

In [2]:
def get_mvp_table(year):
    url = f"https://www.basketball-reference.com/awards/awards_{year}.html"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.basketball-reference.com/awards/"
    }
    
    response = requests.get(url, headers=headers)
    
    print(year, response.status_code)
    
    tables = pd.read_html(StringIO(response.text)) #asks pandas to find all tables inside webpage
    #StringIO makes it behave like a readable file for pandas
    mvp_table = tables[0]
    
    if isinstance(mvp_table.columns, pd.MultiIndex):
        mvp_table.columns = mvp_table.columns.droplevel(0)
    
    mvp_table["Season"] = year
    
    return mvp_table

## Scraping MVP Voting Data

The first dataset comes from Basketball Reference's MVP voting tables. Each seasons awards page contains the players who received MVP votes, along with their vote share and season statistics.

In [3]:
mvp_2025 = get_mvp_table(2025)
mvp_2025.head(10)

2025 200


,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,MP,...,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48,Season
0,1,Shai Gilgeous-Alexander,26,OKC,71,913,1000,0.913,76,34.2,...,5.0,6.4,1.7,1.0,0.519,0.375,0.898,16.7,0.309,2025
1,2,Nikola JokiÄ,29,DEN,29,787,1000,0.787,70,36.7,...,12.7,10.2,1.8,0.6,0.576,0.417,0.800,16.4,0.307,2025
2,3,Giannis Antetokounmpo,30,MIL,0,470,1000,0.470,67,34.2,...,11.9,6.5,0.9,1.2,0.601,0.222,0.617,11.5,0.241,2025
3,4,Jayson Tatum,26,BOS,0,311,1000,0.311,72,36.4,...,8.7,6.0,1.1,0.5,0.452,0.343,0.814,9.5,0.174,2025
4,5,Donovan Mitchell,28,CLE,0,74,1000,0.074,71,31.4,...,4.5,5.0,1.3,0.2,0.443,0.368,0.823,7.6,0.163,2025
5,6,LeBron James,40,LAL,0,16,1000,0.016,70,34.9,...,7.8,8.2,1.0,0.6,0.513,0.376,0.782,7.7,0.152,2025
6,7T,Cade Cunningham,23,DET,0,12,1000,0.012,70,35.0,...,6.1,9.1,1.0,0.8,0.469,0.356,0.846,5.9,0.115,2025
7,7T,Anthony Edwards,23,MIN,0,12,1000,0.012,79,36.3,...,5.7,4.5,1.2,0.6,0.447,0.395,0.837,8.4,0.140,2025
8,9,Stephen Curry,36,GSW,0,2,1000,0.002,70,32.2,...,4.4,6.0,1.1,0.4,0.448,0.397,0.933,7.9,0.168,2025
9,10T,Jalen Brunson,28,NYK,0,1,1000,0.001,65,35.4,...,2.9,7.3,0.9,0.1,0.488,0.383,0.821,8.3,0.172,2025


### Building the MVP Voting Dataset

The following loop collects MVP voting tables from 2010 through 2026 and combines them in to a single dataframe. Each row represents a player who received MVP votes in a given season. NOTE that Basketball Reference is sometimes blocked so I had to add a time.sleep, hence running this cell takes a minute (by design)

In [ ]:
years = range(2010, 2027)

all_mvp_tables = []

for year in years:
    table = get_mvp_table(year)
    all_mvp_tables.append(table)
    time.sleep(6)

all_mvp = pd.concat(all_mvp_tables, ignore_index=True)

all_mvp.head()

In [ ]:
all_mvp.to_csv("all_mvp_2010_2026.csv", index=False) #to save incase blocked

## Selecting Model Variables

From the full MVP voting table I pselected player statistics, efficiency metrics, win-share metrics, vote share, and season. The target variable in the regression (the Y) is MVP vote share.

In [35]:
model_cols = [
    "Player", "Age", "Tm", "G", "MP",
    "PTS", "TRB", "AST", "STL", "BLK",
    "FG%", "3P%", "FT%", "WS", "WS/48",
    "Share", "Season"
]

mvp_model_df = all_mvp[model_cols].copy()

mvp_model_df.head()

,Player,Age,Tm,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48,Share,Season
0,LeBron James,25,CLE,76,39.0,29.7,7.3,8.6,1.6,1.0,0.503,0.333,0.767,18.5,0.299,0.980,2010
1,Kevin Durant,21,OKC,82,39.5,30.1,7.6,2.8,1.4,1.0,0.476,0.365,0.900,16.1,0.238,0.495,2010
2,Kobe Bryant,31,LAL,73,38.8,27.0,5.4,5.0,1.5,0.3,0.456,0.329,0.811,9.4,0.160,0.487,2010
3,Dwight Howard,24,ORL,82,34.7,18.3,13.2,1.8,0.9,2.8,0.612,0.000,0.592,13.2,0.223,0.389,2010
4,Dwyane Wade,28,MIA,77,36.3,26.6,4.8,6.5,1.8,1.1,0.476,0.300,0.761,13.0,0.224,0.097,2010


In [36]:
#checks data types bc sometimes scraped tables have "numbers" stored as text
mvp_model_df.dtypes

Player     object
Age         int64
Tm         object
G           int64
MP        float64
PTS       float64
TRB       float64
AST       float64
STL       float64
BLK       float64
FG%       float64
3P%       float64
FT%       float64
WS        float64
WS/48     float64
Share     float64
Season      int64
dtype: object

In [37]:
#corrects missing values and confirms cleanliness
mvp_model_df = mvp_model_df.dropna().copy()
mvp_model_df.isna().sum()

Player    0
Age       0
Tm        0
G         0
MP        0
PTS       0
TRB       0
AST       0
STL       0
BLK       0
FG%       0
3P%       0
FT%       0
WS        0
WS/48     0
Share     0
Season    0
dtype: int64

## Model 1: First Statistical Regression (Broad)

The first model uses a larger set of player statistics including scoring, efficiency, availability, and winshare metrics. This version gives the model a broad statistical view of each MVP candidate, but many of these variables overlap with each other which makes practical results annoying.

In [38]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

features = [
    "Age", "G", "MP",
    "PTS", "TRB", "AST", "STL", "BLK",
    "FG%", "3P%", "FT%", "WS", "WS/48"
]

X = mvp_model_df[features]
y = mvp_model_df["Share"]

model = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

model.fit(X, y); #this is where python solves the least squares problem

#so our model is taking stat columns in X, standardizing them, feeding them into linear regression, learning
#coefficients to predict share

In [39]:
coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.named_steps["linearregression"].coef_
})

coef_df["Abs Coefficient"] = coef_df["Coefficient"].abs()

coef_df.sort_values("Abs Coefficient", ascending=False)

,Feature,Coefficient,Abs Coefficient
11,WS,0.334546,0.334546
3,PTS,0.095861,0.095861
2,MP,-0.090109,0.090109
1,G,-0.087284,0.087284
12,WS/48,-0.083773,0.083773
10,FT%,-0.063387,0.063387
8,FG%,-0.030657,0.030657
5,AST,0.027660,0.027660
4,TRB,0.016484,0.016484
6,STL,-0.011421,0.011421


Because several features measure very interlinked concepts (such as scoring volume, efficiency, win shares, etc) so the coefficients should not be interpreted as pure isolated effects. Later models in the notebook will clean this up.

In [40]:
mvp_model_df["Predicted Share"] = model.predict(X) 
#^uses trained model to predict mvp vote share for each player

mvp_model_df["Prediction Error"] = mvp_model_df["Share"] - mvp_model_df["Predicted Share"]

mvp_model_df[mvp_model_df["Season"] == 2026][[
    "Player", "Season", "Share", "Predicted Share", "Prediction Error"
]].sort_values("Predicted Share", ascending=False)

#Generate Model 1 predictions for each MVP vote getter

,Player,Season,Share,Predicted Share,Prediction Error
208,Nikola JokiÄ,2026,0.634,0.631522,0.002478
207,Shai Gilgeous-Alexander,2026,0.939,0.620438,0.318562
210,Luka DonÄiÄ,2026,0.250,0.367680,-0.117680
209,Victor Wembanyama,2026,0.569,0.284989,0.284011
213,Kawhi Leonard,2026,0.001,0.177095,-0.176095
211,Cade Cunningham,2026,0.117,0.112646,0.004354
214,Donovan Mitchell,2026,0.001,0.089064,-0.088064
212,Jaylen Brown,2026,0.089,0.056046,0.032954


In [41]:
mvp_model_df[mvp_model_df["Season"] == 2025][[
    "Player", "Share", "Predicted Share", "Prediction Error"
]].sort_values("Predicted Share", ascending=False)

#how does it predict 2025?

,Player,Share,Predicted Share,Prediction Error
196,Nikola JokiÄ,0.787,0.722663,0.064337
195,Shai Gilgeous-Alexander,0.913,0.715887,0.197113
197,Giannis Antetokounmpo,0.470,0.560821,-0.090821
198,Jayson Tatum,0.311,0.161376,0.149624
204,Jalen Brunson,0.001,0.091890,-0.090890
206,Evan Mobley,0.001,0.089167,-0.088167
199,Donovan Mitchell,0.074,0.085217,-0.011217
200,LeBron James,0.016,0.065550,-0.049550
203,Stephen Curry,0.002,0.036115,-0.034115
202,Anthony Edwards,0.012,-0.015949,0.027949


We can see issues here for example. Model 1 slightly favored Jokic over Shai in 2025 even though the actual vote heavily favored Shai. This suggests that the broad stat only model may overweight all around statistical profiles.

(also note I began this a day or two before 2026 MVP results released but '25 and '26 do show similar Jokic-biased errors which makes some sense)

Next step is to train on previous seasons and test on a held out season instead

In [42]:
train_df = mvp_model_df[mvp_model_df["Season"] < 2026].copy()
test_df = mvp_model_df[mvp_model_df["Season"] == 2026].copy()

X_train = train_df[features]
y_train = train_df["Share"]

X_test = test_df[features]
y_test = test_df["Share"]

model_v1_test = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

model_v1_test.fit(X_train, y_train);

test_df["Predicted Share"] = model_v1_test.predict(X_test)
test_df["Prediction Error"] = test_df["Share"] - test_df["Predicted Share"]

test_df[[
    "Player", "Share", "Predicted Share", "Prediction Error"
]].sort_values("Predicted Share", ascending=False)

,Player,Share,Predicted Share,Prediction Error
208,Nikola JokiÄ,0.634,0.619437,0.014563
207,Shai Gilgeous-Alexander,0.939,0.587651,0.351349
210,Luka DonÄiÄ,0.250,0.370978,-0.120978
209,Victor Wembanyama,0.569,0.215835,0.353165
213,Kawhi Leonard,0.001,0.201224,-0.200224
214,Donovan Mitchell,0.001,0.101234,-0.100234
211,Cade Cunningham,0.117,0.098040,0.018960
212,Jaylen Brown,0.089,0.069576,0.019424


## Model 2: Cleaner Production-Based Feature Set

Model 2 uses a smaller set of core player production variables. The goal is to reduce overlap between similar statistics while still capturing scoring, playmaking, defense, rebounding, and overall impact.

In [43]:
features_v2 = [
    "PTS", "TRB", "AST", "STL", "BLK",
    "WS"
]

X_train_v2 = train_df[features_v2]
y_train_v2 = train_df["Share"]

X_test_v2 = test_df[features_v2]
y_test_v2 = test_df["Share"]

model_v2_test = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

model_v2_test.fit(X_train_v2, y_train_v2)

test_df["Predicted Share V2"] = model_v2_test.predict(X_test_v2)
test_df["Prediction Error V2"] = test_df["Share"] - test_df["Predicted Share V2"]

test_df[[
    "Player", "Share", "Predicted Share V2", "Prediction Error V2"
]].sort_values("Predicted Share V2", ascending=False)

,Player,Share,Predicted Share V2,Prediction Error V2
208,Nikola JokiÄ,0.634,0.651785,-0.017785
207,Shai Gilgeous-Alexander,0.939,0.508146,0.430854
210,Luka DonÄiÄ,0.250,0.336297,-0.086297
209,Victor Wembanyama,0.569,0.212663,0.356337
213,Kawhi Leonard,0.001,0.136361,-0.135361
211,Cade Cunningham,0.117,0.107473,0.009527
214,Donovan Mitchell,0.001,0.089375,-0.088375
212,Jaylen Brown,0.089,0.059020,0.029980


In [44]:
from sklearn.metrics import mean_absolute_error

def backtest_model(feature_list, start_year=2015, end_year=2026):
    results = []

    for season in range(start_year, end_year + 1):
        train = mvp_model_df[mvp_model_df["Season"] < season].copy()
        test = mvp_model_df[mvp_model_df["Season"] == season].copy()

        X_train = train[feature_list]
        y_train = train["Share"]
        X_test = test[feature_list]

        model = make_pipeline(
            StandardScaler(),
            LinearRegression()
        )

        model.fit(X_train, y_train)

        test["Predicted Share"] = model.predict(X_test)

        actual_winner = test.sort_values("Share", ascending=False).iloc[0]
        predicted_winner = test.sort_values("Predicted Share", ascending=False).iloc[0]

        results.append({
            "Season": season,
            "Actual MVP": actual_winner["Player"],
            "Predicted MVP": predicted_winner["Player"],
            "Correct": actual_winner["Player"] == predicted_winner["Player"],
            "MAE": mean_absolute_error(test["Share"], test["Predicted Share"])
        })

    return pd.DataFrame(results)

In [45]:
v2_results = backtest_model(features_v2)

v2_results

,Season,Actual MVP,Predicted MVP,Correct,MAE
0,2015,Stephen Curry,James Harden,False,0.174053
1,2016,Stephen Curry,Stephen Curry,True,0.236208
2,2017,Russell Westbrook,James Harden,False,0.232902
3,2018,James Harden,James Harden,True,0.148384
4,2019,Giannis Antetokounmpo,James Harden,False,0.153523
5,2020,Giannis Antetokounmpo,James Harden,False,0.178917
6,2021,Nikola JokiÄ,Nikola JokiÄ,True,0.169800
7,2022,Nikola JokiÄ,Nikola JokiÄ,True,0.147083
8,2023,Joel Embiid,Nikola JokiÄ,False,0.184103
9,2024,Nikola JokiÄ,Nikola JokiÄ,True,0.173516


In [46]:
print("Model 2 Results")
print("------------------------")
print("Correct rate:", round(v2_results["Correct"].mean(), 3))
print("Average MAE:", round(v2_results["MAE"].mean(), 3))

Model 2 Results
------------------------
Correct rate: 0.417
Average MAE: 0.169


## Model 3: Adding Team Success

Model 3 adds team winning percentage as a predictor. MVP voting often reflects both individual production and team success, which checks out. NOTE once again the neccessary time.sleep makes this cell take a minute to run so Basketball Reference doesn't throw issues

In [47]:
team_name_to_abbrev = {
    "Atlanta Hawks": "ATL",
    "Boston Celtics": "BOS",
    "Brooklyn Nets": "BRK",
    "New Jersey Nets": "NJN",
    "Charlotte Bobcats": "CHA",
    "Charlotte Hornets": "CHO",
    "Chicago Bulls": "CHI",
    "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL",
    "Denver Nuggets": "DEN",
    "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW",
    "Houston Rockets": "HOU",
    "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC",
    "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA",
    "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN",
    "New Orleans Hornets": "NOH",
    "New Orleans Pelicans": "NOP",
    "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL",
    "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHO",
    "Portland Trail Blazers": "POR",
    "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR",
    "Utah Jazz": "UTA",
    "Washington Wizards": "WAS"
}

def get_team_records(year):
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}_standings.html"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.basketball-reference.com/"
    }
    
    response = requests.get(url, headers=headers)
    print(year, response.status_code)
    
    tables = pd.read_html(StringIO(response.text))
    
    east = tables[0]
    west = tables[1]
    
    east = east.rename(columns={east.columns[0]: "Team"})
    west = west.rename(columns={west.columns[0]: "Team"})
    
    records = pd.concat([east, west], ignore_index=True)
    
    records["Team"] = (
        records["Team"]
        .astype(str)
        .str.replace("\xa0", " ", regex=False)
        .str.replace("*", "", regex=False)
        .str.replace(r"\s*\(\d+\)", "", regex=True)
        .str.strip()
    )
    
    records["Tm"] = records["Team"].map(team_name_to_abbrev)
    records["Season"] = year
    
    records = records.dropna(subset=["Tm"]).copy()
    
    records = records[["Season", "Tm", "W", "L", "W/L%"]].copy()
    
    records["W"] = pd.to_numeric(records["W"], errors="coerce")
    records["L"] = pd.to_numeric(records["L"], errors="coerce")
    records["W/L%"] = pd.to_numeric(records["W/L%"], errors="coerce")
    
    return records


all_team_records = []

for year in range(2010, 2027):
    records = get_team_records(year)
    all_team_records.append(records)
    time.sleep(6)

team_records = pd.concat(all_team_records, ignore_index=True)

2010 200
2011 200
2012 200
2013 200
2014 200
2015 200
2016 200
2017 200
2018 200
2019 200
2020 200
2021 200
2022 200
2023 200
2024 200
2025 200
2026 200


In [48]:
#merging mvp_model_df with team_records and matching rows where season and team abbrev match
mvp_model_df_v3 = mvp_model_df[model_cols].merge(
    team_records[["Season", "Tm", "W/L%"]],
    on=["Season", "Tm"],
    how="left"
)

mvp_model_df_v3 = mvp_model_df_v3.dropna(subset=["W/L%"]).copy()

mvp_model_df_v3.head()

,Player,Age,Tm,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48,Share,Season,W/L%
0,LeBron James,25,CLE,76,39.0,29.7,7.3,8.6,1.6,1.0,0.503,0.333,0.767,18.5,0.299,0.980,2010,0.744
1,Kevin Durant,21,OKC,82,39.5,30.1,7.6,2.8,1.4,1.0,0.476,0.365,0.900,16.1,0.238,0.495,2010,0.610
2,Kobe Bryant,31,LAL,73,38.8,27.0,5.4,5.0,1.5,0.3,0.456,0.329,0.811,9.4,0.160,0.487,2010,0.695
3,Dwight Howard,24,ORL,82,34.7,18.3,13.2,1.8,0.9,2.8,0.612,0.000,0.592,13.2,0.223,0.389,2010,0.720
4,Dwyane Wade,28,MIA,77,36.3,26.6,4.8,6.5,1.8,1.1,0.476,0.300,0.761,13.0,0.224,0.097,2010,0.573


In [49]:
# quick name cleanup for display
def clean_player_names(df):
    df["Player"] = df["Player"].astype(str)
    df["Player"] = df["Player"].str.replace(r"Nikola Joki.*", "Nikola Jokic", regex=True)
    df["Player"] = df["Player"].str.replace(r"Luka Don.*", "Luka Doncic", regex=True)
    df["Player"] = df["Player"].str.replace(r"Manu Gin.*", "Manu Ginobili", regex=True)
    return df

mvp_model_df = clean_player_names(mvp_model_df)
mvp_model_df_v3 = clean_player_names(mvp_model_df_v3)

## Final Model 3 Backtest

The final version of the model uses core player production statistics plus team winning percentage. It's backtested from 2015 through 2026 by training only on seasons before the year being tested.

In [ ]:
def backtest_model_v3(feature_list, start_year=2015, end_year=2026):
    results = []

    for season in range(start_year, end_year + 1):
        train = mvp_model_df_v3[mvp_model_df_v3["Season"] < season].copy()
        test = mvp_model_df_v3[mvp_model_df_v3["Season"] == season].copy()

        X_train = train[feature_list]
        y_train = train["Share"]
        X_test = test[feature_list]

        model = make_pipeline(
            StandardScaler(),
            LinearRegression()
        )

        model.fit(X_train, y_train)

        test["Predicted Share"] = model.predict(X_test)

        actual_winner = test.sort_values("Share", ascending=False).iloc[0]
        predicted_winner = test.sort_values("Predicted Share", ascending=False).iloc[0]

        results.append({
            "Season": season,
            "Actual MVP": actual_winner["Player"],
            "Predicted MVP": predicted_winner["Player"],
            "Correct": actual_winner["Player"] == predicted_winner["Player"],
            "MAE": mean_absolute_error(test["Share"], test["Predicted Share"])
        })

    return pd.DataFrame(results)

features_v3 = [
    "PTS", "TRB", "AST", "STL", "BLK",
    "WS", "W/L%"
]

v3_results = backtest_model_v3(features_v3, start_year=2015, end_year=2026)

model3_correct = v3_results["Correct"].mean()
model3_avg_mae = v3_results["MAE"].mean()
model3_correct_count = v3_results["Correct"].sum()
model3_total = len(v3_results)

print("Model 3 Results")
print("----------------")
print("Correct MVP picks:", model3_correct_count, "out of", model3_total)
print("Correct rate:", round(model3_correct, 3))
print("Average MAE:", round(model3_avg_mae, 3))

display(v3_results)

# final 2026 prediction table: train through 2025, test on 2026
train_v3_final = mvp_model_df_v3[mvp_model_df_v3["Season"] < 2026].copy()
test_2026 = mvp_model_df_v3[mvp_model_df_v3["Season"] == 2026].copy()

X_train_v3_final = train_v3_final[features_v3]
y_train_v3_final = train_v3_final["Share"]

model_v3_final = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

model_v3_final.fit(X_train_v3_final, y_train_v3_final)

test_2026["Predicted Share"] = model_v3_final.predict(test_2026[features_v3])
test_2026["Prediction Error"] = test_2026["Share"] - test_2026["Predicted Share"]

display(
    test_2026[
        ["Player", "Tm", "Share", "Predicted Share", "Prediction Error"]
    ].sort_values("Predicted Share", ascending=False)
)

coef_v3_df = pd.DataFrame({
    "Feature": features_v3,
    "Coefficient": model_v3_final.named_steps["linearregression"].coef_
})

coef_v3_df["Abs Coefficient"] = coef_v3_df["Coefficient"].abs()

display(coef_v3_df.sort_values("Abs Coefficient", ascending=False))